# 06 - التصنيف الكامل والعزل (Classification & Quarantine)

هذا النوتبوك يغطي القسم **6.6** (التصنيف) و **6.8** (Quarantine) و **6.9** (المجموعات) و **6.10** (Upsert + Business Key) من وثيقة التكليف.

**ماذا يحدث هنا:**
1. يمر على **كامل** بيانات `orders_raw` (~30.1 مليون سجل، من كلا المحركين)
2. يطبّق `classify_and_clean` على كل سجل
3. يكتب السجلات السليمة/المصححة إلى `orders_validated` عبر **Upsert** حسب `order_id` (Business Key ثابت — القسم 6.10)
4. يكتب السجلات المعزولة إلى `orders_quarantine` مع `codes_error` و `details_error` والسجل الخام كاملاً
5. ينشئ **فهرس فريد (Unique Index)** على `order_id` داخل `orders_validated`

⚠️ **تنبيه مهم جدًا بخصوص الوقت:** معالجة 30.1 مليون سجل بمعالج Python تسلسلي (حتى مع الكتابة الجماعية Bulk Write) ستأخذ **وقتًا طويلاً** — على الأرجح **ساعة أو أكثر** حسب سرعة جهازك. الكود يطبع تقدمًا دوريًا مع تقدير الوقت المتبقي — راقبه واتركه يعمل بالخلفية دون مقاطعة.

**ملاحظة تصميمية مهمة:** العينة الصغيرة (`orders_small_sample.csv`) هي فعليًا أول 100,000 سطر من نفس الملف الضخم (استُخرجت بوضع `head`). هذا يعني أن نفس `order_id` سيظهر مرتين في `orders_raw` (مرة من Python Batch، ومرة من PySpark). هذا **مقصود ومفيد**: يثبت عمليًا منطق `ID_ORDER_DUPLICATE` — أول ظهور لكل `order_id` يُعالَج عاديًا، والتكرارات اللاحقة تُعزل تلقائيًا.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / "src"))

from config import settings
from elt_pipeline import run_full_classification
from pymongo import MongoClient

client = MongoClient(settings.MONGO_URI)
db = client[settings.MONGO_DB_NAME]

total_raw = db[settings.COLLECTION_RAW].count_documents({})
print(f"إجمالي السجلات في orders_raw (سيتم تصنيفها بالكامل): {total_raw:,}")
client.close()

إجمالي السجلات في orders_raw (سيتم تصنيفها بالكامل): 30,100,000


## تشغيل التصنيف الكامل

هذه الخلية هي الأثقل بالمشروع بالكامل. راقب رسائل التقدم أسفلها.

In [2]:
classification_summary = run_full_classification(
    batch_size=5000,
    progress_every_batches=20,
)

print("\n\n== ملخص التصنيف الكامل ==")
for k, v in classification_summary.items():
    if k not in ("quarantine_code_counts", "correction_rule_counts"):
        print(f"{k}: {v}")

[OK] فهرس فريد على order_id داخل orders_validated جاهز.
إجمالي السجلات المستهدفة للتصنيف: 30,100,000

تقدم: 100,000/30,100,000 (0.3%) | معدل: 5,675 سجل/ثانية | الزمن المنقضي: 0.3 دقيقة | الوقت المتبقي التقديري: 88.1 دقيقة
تقدم: 200,000/30,100,000 (0.7%) | معدل: 7,700 سجل/ثانية | الزمن المنقضي: 0.4 دقيقة | الوقت المتبقي التقديري: 64.7 دقيقة
تقدم: 300,000/30,100,000 (1.0%) | معدل: 6,839 سجل/ثانية | الزمن المنقضي: 0.7 دقيقة | الوقت المتبقي التقديري: 72.6 دقيقة
تقدم: 400,000/30,100,000 (1.3%) | معدل: 6,526 سجل/ثانية | الزمن المنقضي: 1.0 دقيقة | الوقت المتبقي التقديري: 75.9 دقيقة
تقدم: 500,000/30,100,000 (1.7%) | معدل: 6,335 سجل/ثانية | الزمن المنقضي: 1.3 دقيقة | الوقت المتبقي التقديري: 77.9 دقيقة
تقدم: 600,000/30,100,000 (2.0%) | معدل: 6,150 سجل/ثانية | الزمن المنقضي: 1.6 دقيقة | الوقت المتبقي التقديري: 80.0 دقيقة
تقدم: 700,000/30,100,000 (2.3%) | معدل: 5,874 سجل/ثانية | الزمن المنقضي: 2.0 دقيقة | الوقت المتبقي التقديري: 83.4 دقيقة
تقدم: 800,000/30,100,000 (2.7%) | معدل: 5,851 سجل/ثانية | 

MongoClient background task encountered an error:
Traceback (most recent call last):
  File "C:\Users\Hashem Hobireh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pymongo\synchronous\mongo_client.py", line 2262, in _process_kill_cursors
    self._kill_cursors(cursor_ids, address, topology, session=None)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Hashem Hobireh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pymongo\synchronous\mongo_client.py", line 2208, in _kill_cursors
    server = topology.select_server_by_address(tuple(address), _Op.KILL_CURSORS)  # type: ignore[arg-type]
  File "C:\Users\Hashem Hobireh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pymongo\synchronous\topology.py", line 477, in select_server_by_address
    return self.select_server(
           ~~~~~~~~~~~~~~~~~~^
        any_server_selector,
        ^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
        operation_id=operation_id,
        ^^^^^^^

AutoReconnect: localhost:27017: [WinError 10054] An existing connection was forcibly closed by the remote host (configured timeouts: connectTimeoutMS: 20000.0ms)

## معادلة الاتساق على كامل البيانات (القسم 6.11)

run_raw_count = run_valid_count + run_corrected_count + run_quarantine_count

In [ ]:
read_rows = classification_summary["read_rows"]
valid = classification_summary["count_valid"]
corrected = classification_summary["count_corrected"]
quarantine = classification_summary["count_quarantine"]

total_check = valid + corrected + quarantine
print(f"read_rows: {read_rows:,}")
print(f"valid + corrected + quarantine = {valid:,} + {corrected:,} + {quarantine:,} = {total_check:,}")

assert total_check == read_rows, "عدم اتساق! معادلة القسم 6.11 غير متحققة."
print("\n[تحقق ناجح] معادلة الاتساق محققة على كامل البيانات (30.1 مليون سجل).")

## التحقق من MongoDB مباشرة

In [ ]:
client = MongoClient(settings.MONGO_URI)
db = client[settings.MONGO_DB_NAME]

validated_count = db[settings.COLLECTION_VALIDATED].count_documents({})
quarantine_count_db = db[settings.COLLECTION_QUARANTINE].count_documents({})

print(f"عدد السجلات الفريدة في orders_validated (بعد Upsert حسب order_id): {validated_count:,}")
print(f"عدد السجلات في orders_quarantine: {quarantine_count_db:,}")
print("\nملاحظة: orders_validated قد يكون عدده أقل من (valid+corrected) بسبب")
print("Upsert لنفس order_id المكرر بين العينة والملف الضخم (سلوك متوقع وصحيح).")

indexes = list(db[settings.COLLECTION_VALIDATED].list_indexes())
print("\nفهارس orders_validated:")
for idx in indexes:
    print(f"  - {idx['name']}: {idx.get('key')}, unique={idx.get('unique', False)}")

client.close()

## توزيع أكواد العزل وقواعد التصحيح على كامل البيانات

In [ ]:
print("== توزيع أكواد العزل (كامل البيانات) ==")
for code, count in sorted(
    classification_summary["quarantine_code_counts"].items(), key=lambda x: -x[1]
):
    print(f"{code}: {count:,}")

print("\n== توزيع قواعد التصحيح المُطبَّقة (كامل البيانات) ==")
for rule, count in sorted(
    classification_summary["correction_rule_counts"].items(), key=lambda x: -x[1]
):
    print(f"{rule}: {count:,}")

## عينة من MongoDB Compass

افتح MongoDB Compass، اتصل بـ mongodb://localhost:27017، وتصفح قاعدة البيانات midterm_pipeline_db لترى المجموعات الثلاث فعليًا: orders_raw, orders_validated, orders_quarantine. هذا مطلوب للتسليم (لقطات Compass، القسم 11).

## الخطوة القادمة

**07_upsert_idempotency.ipynb** — إثبات Idempotency: إعادة تشغيل نفس الملف مرة ثانية والتحقق من أن orders_validated لا يزيد عددًا، ثم تعديل سجل واحد وإظهار أنه يُحدَّث (Update) دون تكرار (Duplicate).